In [1]:
"""
Centroid and Moment of Inertia Calculator
==========================================
Computes centroids and area moments of inertia for composite sections
(I-beams, T-beams, channels, or any combination of rectangles/circles)
using the parallel axis theorem.

Sign convention: holes / cutouts are entered as NEGATIVE area shapes and
subtract automatically from area, centroid, and inertia calculations.

Dependencies:
    pip install numpy matplotlib
"""

from dataclasses import dataclass
from typing import List, Optional
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


# --------------------------------------------------------------------------
# Basic shape primitive
# --------------------------------------------------------------------------

@dataclass
class Shape:
    name: str
    area: float      # negative area = hole / cutout
    cx: float         # centroid x location (global coords)
    cy: float         # centroid y location (global coords)
    Ixx_c: float      # moment of inertia about its OWN centroidal x-axis
    Iyy_c: float      # moment of inertia about its OWN centroidal y-axis
    Ixy_c: float = 0.0
    # for plotting only:
    kind: str = "rect"        # "rect" or "circle"
    dims: Optional[tuple] = None   # (width, height) for rect, (radius,) for circle


# ---------------- shape factory helpers ----------------

def make_rectangle(width: float, height: float, cx: float, cy: float,
                    name: str = "Rect", hole: bool = False) -> Shape:
    """cx, cy = centroid location of the rectangle in GLOBAL coordinates."""
    area = width * height * (-1 if hole else 1)
    Ixx_c = width * height ** 3 / 12
    Iyy_c = height * width ** 3 / 12
    return Shape(name, area, cx, cy, Ixx_c, Iyy_c, kind="rect", dims=(width, height))


def make_circle(radius: float, cx: float, cy: float,
                 name: str = "Circle", hole: bool = False) -> Shape:
    area = np.pi * radius ** 2 * (-1 if hole else 1)
    Ixx_c = Iyy_c = np.pi * radius ** 4 / 4
    return Shape(name, area, cx, cy, Ixx_c, Iyy_c, kind="circle", dims=(radius,))


def make_triangle_right(base: float, height: float, cx_apex: float, cy_apex: float,
                         name: str = "Triangle") -> Shape:
    """
    Right triangle with the right angle at (cx_apex, cy_apex), base extending
    in +x, height extending in +y. Centroid is offset base/3 and height/3
    from the right-angle corner.
    """
    area = 0.5 * base * height
    cx = cx_apex + base / 3
    cy = cy_apex + height / 3
    Ixx_c = base * height ** 3 / 36
    Iyy_c = height * base ** 3 / 36
    return Shape(name, area, cx, cy, Ixx_c, Iyy_c, kind="rect", dims=(base, height))


# ---------------- common composite cross-sections ----------------

def make_i_beam(bf: float, tf: float, d: float, tw: float,
                origin=(0.0, 0.0)) -> List[Shape]:
    """
    Standard I-beam (wide-flange) built from 3 rectangles.
      bf = flange width, tf = flange thickness
      d  = total depth (top of top flange to bottom of bottom flange)
      tw = web thickness
    `origin` = bottom-left corner of the overall bounding box.
    """
    ox, oy = origin
    web_h = d - 2 * tf

    bottom_flange = make_rectangle(bf, tf, ox + bf / 2, oy + tf / 2, "Bottom Flange")
    web = make_rectangle(tw, web_h, ox + bf / 2, oy + tf + web_h / 2, "Web")
    top_flange = make_rectangle(bf, tf, ox + bf / 2, oy + tf + web_h + tf / 2, "Top Flange")
    return [bottom_flange, web, top_flange]


def make_t_beam(bf: float, tf: float, d: float, tw: float,
                origin=(0.0, 0.0)) -> List[Shape]:
    """
    T-beam: one flange on top, web below.
      bf = flange width, tf = flange thickness
      d  = total depth, tw = web thickness
    `origin` = bottom-left corner of the web.
    """
    ox, oy = origin
    web_h = d - tf
    web = make_rectangle(tw, web_h, ox + tw / 2, oy + web_h / 2, "Web")
    flange = make_rectangle(bf, tf, ox + tw / 2, oy + web_h + tf / 2, "Flange")
    return [web, flange]


def make_channel(bf: float, tf: float, d: float, tw: float,
                  origin=(0.0, 0.0)) -> List[Shape]:
    """
    C-channel: web on the left, flanges top & bottom extending to the right.
      bf = flange width (measured from the web's left face)
      tf = flange thickness, d = total depth, tw = web thickness
    `origin` = bottom-left corner of the overall bounding box.
    """
    ox, oy = origin
    web_h = d - 2 * tf

    web = make_rectangle(tw, d, ox + tw / 2, oy + d / 2, "Web")
    bottom_flange = make_rectangle(bf, tf, ox + bf / 2, oy + tf / 2, "Bottom Flange")
    top_flange = make_rectangle(bf, tf, ox + bf / 2, oy + d - tf / 2, "Top Flange")
    return [web, bottom_flange, top_flange]


# --------------------------------------------------------------------------
# Composite section solver
# --------------------------------------------------------------------------

class CompositeSection:
    def __init__(self, parts: List[Shape]):
        self.parts = parts

    @property
    def total_area(self) -> float:
        return sum(p.area for p in self.parts)

    def centroid(self) -> tuple:
        A = self.total_area
        x_bar = sum(p.area * p.cx for p in self.parts) / A
        y_bar = sum(p.area * p.cy for p in self.parts) / A
        return x_bar, y_bar

    def moments_of_inertia(self) -> dict:
        """
        Returns Ixx, Iyy, Ixy about the COMPOSITE centroid, using the
        parallel axis theorem:  I = I_own + A * d^2
        """
        x_bar, y_bar = self.centroid()
        Ixx = Iyy = Ixy = 0.0
        for p in self.parts:
            dx = p.cx - x_bar
            dy = p.cy - y_bar
            Ixx += p.Ixx_c + p.area * dy ** 2
            Iyy += p.Iyy_c + p.area * dx ** 2
            Ixy += p.Ixy_c + p.area * dx * dy
        return {"Ixx": Ixx, "Iyy": Iyy, "Ixy": Ixy}

    def radii_of_gyration(self) -> dict:
        I = self.moments_of_inertia()
        A = self.total_area
        return {"rx": np.sqrt(I["Ixx"] / A), "ry": np.sqrt(I["Iyy"] / A)}

    def summary(self) -> str:
        x_bar, y_bar = self.centroid()
        I = self.moments_of_inertia()
        r = self.radii_of_gyration()
        lines = [
            f"Total area:        {self.total_area:.4f}",
            f"Centroid (x, y):   ({x_bar:.4f}, {y_bar:.4f})",
            f"Ixx (about centroid): {I['Ixx']:.4f}",
            f"Iyy (about centroid): {I['Iyy']:.4f}",
            f"Ixy (about centroid): {I['Ixy']:.4f}",
            f"rx (radius of gyration): {r['rx']:.4f}",
            f"ry (radius of gyration): {r['ry']:.4f}",
        ]
        return "\n".join(lines)

    # ---------------- Visualization ----------------
    def plot(self, title="Composite Section", show=True, savepath=None):
        fig, ax = plt.subplots(figsize=(7, 7))

        for p in self.parts:
            is_hole = p.area < 0
            face = "white" if is_hole else "steelblue"
            edge = "crimson" if is_hole else "black"
            alpha = 1.0 if is_hole else 0.6

            if p.kind == "rect":
                w, h = p.dims
                ax.add_patch(patches.Rectangle((p.cx - w / 2, p.cy - h / 2), w, h,
                                                facecolor=face, edgecolor=edge,
                                                lw=1.5, alpha=alpha, zorder=1))
            elif p.kind == "circle":
                (r,) = p.dims
                ax.add_patch(patches.Circle((p.cx, p.cy), r,
                                             facecolor=face, edgecolor=edge,
                                             lw=1.5, alpha=alpha, zorder=1))

            ax.plot(p.cx, p.cy, "k+", ms=8, zorder=2)
            ax.annotate(p.name, (p.cx, p.cy), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, zorder=3)

        x_bar, y_bar = self.centroid()
        ax.plot(x_bar, y_bar, marker="o", color="red", ms=10, zorder=4,
                label=f"Centroid ({x_bar:.3f}, {y_bar:.3f})")
        ax.axhline(y_bar, color="red", ls="--", lw=1, alpha=0.6)
        ax.axvline(x_bar, color="red", ls="--", lw=1, alpha=0.6)

        ax.set_aspect("equal")
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper right", fontsize=9)

        if savepath:
            plt.savefig(savepath, dpi=150, bbox_inches="tight")
        if show:
            plt.show()
        plt.close()


# --------------------------------------------------------------------------
# Demo
# --------------------------------------------------------------------------

if __name__ == "__main__":
    # --- I-beam example ---
    print("=== I-Beam ===")
    i_beam_parts = make_i_beam(bf=6, tf=0.5, d=10, tw=0.4)
    i_beam = CompositeSection(i_beam_parts)
    print(i_beam.summary())
    i_beam.plot(title="I-Beam Cross Section", show=False, savepath="i_beam.png")
    print()

    # --- T-beam example ---
    print("=== T-Beam ===")
    t_beam_parts = make_t_beam(bf=8, tf=0.75, d=10, tw=0.5)
    t_beam = CompositeSection(t_beam_parts)
    print(t_beam.summary())
    t_beam.plot(title="T-Beam Cross Section", show=False, savepath="t_beam.png")
    print()

    # --- Channel example ---
    print("=== Channel (C-Section) ===")
    channel_parts = make_channel(bf=4, tf=0.5, d=8, tw=0.4)
    channel = CompositeSection(channel_parts)
    print(channel.summary())
    channel.plot(title="Channel Cross Section", show=False, savepath="channel.png")
    print()

    # --- Custom composite example: rectangle with a circular hole ---
    print("=== Rectangle with Circular Hole ===")
    custom_parts = [
        make_rectangle(width=10, height=6, cx=5, cy=3, name="Base Plate"),
        make_circle(radius=1.5, cx=5, cy=3, name="Hole", hole=True),
    ]
    custom = CompositeSection(custom_parts)
    print(custom.summary())
    custom.plot(title="Plate with Hole", show=False, savepath="plate_with_hole.png")

=== I-Beam ===
Total area:        9.6000
Centroid (x, y):   (3.0000, 5.0000)
Ixx (about centroid): 159.8000
Iyy (about centroid): 18.0480
Ixy (about centroid): 0.0000
rx (radius of gyration): 4.0799
ry (radius of gyration): 1.3711

=== T-Beam ===
Total area:        10.6250
Centroid (x, y):   (0.2500, 7.4485)
Ixx (about centroid): 98.5526
Iyy (about centroid): 32.0964
Ixy (about centroid): 0.0000
rx (radius of gyration): 3.0456
ry (radius of gyration): 1.7381

=== Channel (C-Section) ===
Total area:        7.2000
Centroid (x, y):   (1.2000, 4.0000)
Ixx (about centroid): 73.4000
Iyy (about centroid): 11.1360
Ixy (about centroid): 0.0000
rx (radius of gyration): 3.1929
ry (radius of gyration): 1.2437

=== Rectangle with Circular Hole ===
Total area:        52.9314
Centroid (x, y):   (5.0000, 3.0000)
Ixx (about centroid): 183.9761
Iyy (about centroid): 503.9761
Ixy (about centroid): 0.0000
rx (radius of gyration): 1.8643
ry (radius of gyration): 3.0857
